# запускать в collab

In [1]:
%pip install -q conllu graphviz requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import json
from conllu import parse
from graphviz import Digraph
from IPython.display import display


In [3]:
from conllu import parse

with open("ru_pud-ud-test.conllu", "r", encoding="utf-8") as f:
    gold_data = f.read()

gold_sentences = parse(gold_data)


In [4]:
target_sentence = (
    "Первой начала плакать одна из езидских женщин , а затем один из ее друзей ."
)

test_sent = None
for sent in gold_sentences:
    sentence_text = " ".join([t["form"] for t in sent])
    if sentence_text == target_sentence:
        test_sent = sent
        break

test_sent


TokenList<Первой, начала, плакать, одна, из, езидских, женщин, ,, а, затем, один, из, ее, друзей, ., metadata={newdoc id: "n01012", sent_id: "n01012003", parallel_id: "pud/n01012003", text: "Первой начала плакать одна из езидских женщин, а затем один из ее друзей.", english_text: "First one of the Yazidi women started crying, then one of her friends."}>

In [5]:
def parse_syntax_api(text, model_name):
    url = "https://lindat.mff.cuni.cz/services/udpipe/api/process"
    params = {
        "model": model_name,
        "data": text,
        "tokenizer": "true",
        "tagger": "true",
        "parser": "true",
        "uda": "true",
    }
    response = requests.post(url, data=params)
    result = response.json()
    return result["result"]


In [6]:
def parse_output(conll):
    sentences = parse(conll)
    return [s.to_tree() for s in sentences]


In [7]:
class Root(object):
    def __init__(self, child):
        self.token = {"form": "", "id": 0, "deprel": "root"}
        self.children = [child]


def draw_tree(tree):
    g = Digraph(format="png")
    queue = [Root(tree)]

    while len(queue) > 0:
        head = queue.pop()
        head_label = head.token["form"] + " (%d)" % head.token["id"]
        queue.extend(head.children)

        for c in head.children:
            c_label = c.token["form"] + " (%d)" % c.token["id"]
            g.edge(head_label, c_label, label=c.token["deprel"])

    return g


In [8]:
def edges_sets(tree):
    edges_labeled = []
    edges_unlabeled = []
    queue = [Root(tree)]

    while len(queue) > 0:
        head = queue.pop()
        head_label = head.token["form"] + " (%d)" % head.token["id"]
        queue.extend(head.children)

        for c in head.children:
            c_label = c.token["form"] + " (%d)" % c.token["id"]
            edges_unlabeled.append((head_label, c_label))
            edges_labeled.append((head_label, c_label, c.token["deprel"]))

    return set(edges_labeled), set(edges_unlabeled)


def accuracy(gold, pred):
    gold_labeled, gold_unlabeled = edges_sets(gold)
    pred_labeled, pred_unlabeled = edges_sets(pred)

    las = len(gold_labeled.intersection(pred_labeled)) / len(gold_labeled)
    uas = len(gold_unlabeled.intersection(pred_unlabeled)) / len(gold_unlabeled)
    return las, uas


def tree_depth(tree):
    depth = 0
    queue = [(tree, 0)]
    while len(queue) > 0:
        head, d = queue.pop(0)
        depth = max(depth, d)
        queue.extend([(c, d + 1) for c in head.children])
    return depth


In [9]:
gold_tree = test_sent.to_tree()
draw_tree(gold_tree)


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [10]:
syntag_conll = parse_syntax_api(target_sentence, "russian-syntagrus")
syntag_tree = parse_output(syntag_conll.strip())[0]

draw_tree(syntag_tree)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [11]:
las_syn, uas_syn = accuracy(gold_tree, syntag_tree)
depth_syn = tree_depth(syntag_tree)

round(uas_syn, 3), round(las_syn, 3), depth_syn


NameError: name 'syntag_tree' is not defined

In [ ]:
taiga_conll = parse_syntax_api(target_sentence, "russian-taiga")
taiga_tree = parse_output(taiga_conll.strip())[0]

draw_tree(taiga_tree)


In [ ]:
las_taiga, uas_taiga = accuracy(gold_tree, taiga_tree)
depth_taiga = tree_depth(taiga_tree)

round(uas_taiga, 3), round(las_taiga, 3), depth_taiga


In [ ]:
print(round(uas_syn, 3))
print(round(las_syn, 3))
print(depth_syn)

print(round(uas_taiga, 3))
print(round(las_taiga, 3))
print(depth_taiga)
